# Notebook 00 — Data Preprocessing

**Dataset**: `merged_cleaned.csv` (scraped from UAV-NIDD raw captures) — **4,854,083 rows, 135 raw columns** (grew from the originally-documented ~803K rows / ~44 columns as more capture sessions were merged in).
**Goal**: Clean, split (70/15/15, time-based per class), scale, and save processed arrays for all downstream notebooks.

## ⚠ Data leakage fix (read this first)
Diagnostic check found that **every class occupies a disjoint absolute time window**
(each attack type was captured as one separate session). A Decision Tree trained on
`frame.time_epoch` ALONE reached macro F1 = 1.0000 — the model was learning *which
capture session a packet came from*, not actual attack behaviour. Removing the 4
frame-level timing columns dropped F1 to a much more realistic 0.8494.

Fix applied below:
1. **Drop leakage features**: `frame.time_epoch`, `frame.time_relative`,
   `frame.time_delta`, `frame.time_delta_displayed` are excluded from the feature
   matrix (kept temporarily, internally, only as a chronological sort key for splitting).
   `tcp.time_delta` / `tcp.time_relative` are KEPT — they describe per-connection
   timing behaviour, not which session/file a packet came from.
2. **Time-based split per class** (instead of random stratified row split): within
   each class, rows are sorted chronologically by `frame.time_epoch` and split
   70/15/15 along that timeline — earliest 70% → train, next 15% → val, latest 15% → test.
   This prevents near-identical adjacent packets from being scattered across train/test.
   *Limitation*: most classes have only one capture session, so this is a temporal
   hold-out (train on the past of that session, test on its future), not a true
   leave-session-out validation — documented here for the writeup.

## ⚠ Memory: chunked CSV reading (read this too)
The raw CSV is large enough (4.85M rows, 135 columns) that loading it in one shot
(`pd.read_csv` on the whole file) drove this process to ~13.7GB resident memory and
got killed by the kernel's OOM-killer on this machine (15GB RAM total) — this was the
real cause of the `KeyboardInterrupt`/hang seen earlier, not a manual interrupt.
Fix: read and clean the CSV in **50,000-row chunks**, casting each chunk straight to a
small float32 array and discarding the raw chunk DataFrame immediately. Peak memory
stays in the hundreds of MB instead of double-digit GB. See Section 1 below.

## Feature retention policy (no missing-value threshold)
An earlier version of this notebook dropped any column with >50% missing values,
which left only 16 features. **This is wrong for this dataset**: most "missing" values
are not bad data — a field like `wlan.fixed.reason_code` is only populated for
802.11 management frames, `udp.srcport` only for UDP packets, etc. Each feature is
legitimately absent outside its own protocol/attack context, which is exactly the
kind of signal the FS/FE comparison in this study is supposed to exploit. Dropping by
missingness threshold would also make K=16 in the FS/FE experiments equivalent to
"keep everything" — defeating the point of measuring dimensionality reduction.

**New policy**: keep every column that is genuinely numeric (verified across the
*entire* file, not a single chunk — see `NATIVE_NUMERIC_COLS` below), fill all
missing values with 0 (NaN means "not applicable to this packet type"), and drop only:
- identifiers / free-text columns (`DROP_ALWAYS`) — non-numeric by nature, e.g. MAC
  addresses, domain names, raw hex payloads, IP-address strings — confirmed via a
  full-file scan (see `/tmp/classify_columns.py` / `/tmp/classify_object_cols.py` from
  the chat session) that these never parse as plain numbers,
- the 4 leakage columns (session-identity leak, see above),
- zero-variance columns (literally constant after fill — checked automatically).

Two hex-formatted columns (`wlan.fixed.reason_code`, `wlan.fc.ds`) were additionally
recovered via `parse_hex()` — they're genuinely numeric (802.11 reason-code / frame-
direction values), just written as `'0x0007'`-style strings, and are directly
attack-relevant (reason codes are what deauth attack tools set). MAC-address and raw
IP-string columns were deliberately **not** label/one-hot encoded — see the note next
to `HEX_NUMERIC` in the config cell for why (session-identity leakage risk, same class
of bug as `frame.time_epoch`).

This keeps **72 raw features** (vs. 16 under the old threshold-based approach), out
of 135 raw columns, 4.85M rows.

## Preprocessing decisions documented here
| Decision | Value | Reason |
|---|---|---|
| Missing-value threshold | **none** (removed) | missingness reflects protocol/attack-type structure, not bad data — see policy note above |
| Drop always | frame.number, frame.time, ip.src, ip.dst, tcp.analysis | identifiers / non-numeric complex fields |
| Drop (leakage) | frame.time_epoch, frame.time_relative, frame.time_delta, frame.time_delta_displayed | proxy for capture session / class label, see above |
| Object→numeric | ip.version, ip.proto, ip.ttl | numeric values stored as strings (tshark comma-joins duplicate field occurrences, e.g. '64,64'); coerce with errors='coerce' then fill 0 |
| Hex→numeric | wlan.fixed.reason_code, wlan.fc.ds | attack-relevant 802.11 fields stored as '0x..' hex strings; parsed via `int(x, 16)` |
| Not encoded (leakage risk) | wlan.bssid/da/sa/ra/ta, arp.*.hw_mac, arp.*.proto_ipv4 | raw device identity -- label/one-hot encoding would leak session identity like frame.time_epoch did; safe derived features (e.g. is-broadcast) left for feature-engineering phase |
| NaN fill strategy | 0 | For protocol features, NaN = 'field does not apply to this packet type' |
| Zero-variance drop | automatic | checked after fill; constant columns removed |
| CSV reading | chunked, 50,000 rows/chunk | avoids OOM on the full 4.85M-row file (see memory note above) |
| Scaler | StandardScaler | fit **only** on train, transform val+test |
| Split | 70 / 15 / 15, time-based per class | earliest→train, middle→val, latest→test (see fix above) |
| Class-imbalance | NOT handled here — handled per-seed inside each experiment notebook |

In [1]:
import warnings
warnings.filterwarnings('ignore')

import os, time, pickle, json
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

print('numpy :', np.__version__)
print('pandas:', pd.__version__)

numpy : 2.4.6
pandas: 3.0.3


In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────────
# Absolute paths -- robust regardless of which directory Jupyter's cwd resolves to.
BASE_DIR          = '/home/hanh/UAV_/'
RAW_CSV           = f'{BASE_DIR}data/merged_cleaned.csv'
OUT_DIR           = f'{BASE_DIR}processed/'
os.makedirs(OUT_DIR, exist_ok=True)

RANDOM_STATE      = 42
TRAIN_RATIO       = 0.70
VAL_RATIO         = 0.15
TEST_RATIO        = 0.15
CHUNK_SIZE        = 50_000        # rows per chunk -- keeps peak memory low on the 4.85M-row file

# Always drop: identifiers and non-numeric complex fields
DROP_ALWAYS = [
    'frame.number',        # row index — not a network feature
    'frame.time',          # raw timestamp string (epoch kept separately)
    'ip.src', 'ip.dst',   # host identifiers, not features
    'tcp.analysis',        # free-text object (e.g. 'Retransmission')
]

# Object columns whose values are actually numeric stored as strings
COERCE_NUMERIC = ['ip.version', 'ip.proto', 'ip.ttl']

# Hex-string columns that are genuinely numeric, just tshark-formatted as '0x...'
# (verified clean -- no comma-multi-value garbage, small fixed value sets). Both
# are directly attack-relevant, not just "numeric leftovers":
#   wlan.fixed.reason_code -- 802.11 deauth/disassoc reason code (e.g. 7 = "Class 3
#     frame from nonassociated STA", the value common deauth tools like aireplay-ng
#     send) -- highly relevant to the De-Authentication class.
#   wlan.fc.ds -- frame control To-DS/From-DS direction bits, relevant to detecting
#     frames with spoofed/unexpected direction.
HEX_NUMERIC = ['wlan.fixed.reason_code', 'wlan.fc.ds']


def parse_hex(series):
    return series.map(lambda x: int(x, 16) if isinstance(x, str) else np.nan)


# ── DATA LEAKAGE FIX ─────────────────────────────────────────────────────────
# Diagnostic finding: each class was captured in one disjoint absolute time
# window. A model trained on frame.time_epoch ALONE reaches macro F1 = 1.0000,
# i.e. it learns "which capture session" instead of attack behaviour.
# These 4 frame-level timing columns are excluded from the FEATURE matrix.
# tcp.time_delta / tcp.time_relative are kept (per-connection timing, not
# session/capture timing -> not a session-identity leak).
LEAKAGE_COLS = [
    'frame.time_epoch',
    'frame.time_relative',
    'frame.time_delta',
    'frame.time_delta_displayed',
]
TIME_KEY_COL = 'frame.time_epoch'   # used ONLY to chronologically order rows for splitting

LABEL_COL = 'label'

# Note: MAC-address fields (wlan.bssid/da/sa/ra/ta, arp.*.hw_mac) and raw IP-string
# fields (arp.*.proto_ipv4) are intentionally NOT label/one-hot encoded here, even
# though they could plausibly carry attack signal (e.g. broadcast destinations in
# DDoS/deauth floods, ARP-spoofed MAC/IP mismatches in MITM). Encoding the raw
# device identity would reintroduce the same session-identity leakage problem
# already fixed for frame.time_epoch above -- the model would learn "this is the
# attacker's MAC in this capture" rather than generalizable attack behaviour.
# Recovering this signal safely needs a derived feature (e.g. is-broadcast flag,
# IP-to-MAC mapping consistency), which is feature ENGINEERING (notebook 03), not
# preprocessing -- left for that phase.

# ── Feature retention policy (no missing-value threshold -- see markdown above) ─
# Columns confirmed numeric (int64/float64) in EVERY chunk across the full
# 4,854,083-row file. Sparse-but-numeric columns (wlan.*, radiotap.*, dns.count.*,
# ...) are intentionally KEPT -- their missingness reflects "this field only
# applies to one protocol/attack type", not bad data. Determined by scanning the
# whole file once (a single chunk's dtype is not reliable: some columns are
# all-NaN, hence float64-by-default, in early chunks but genuinely textual once
# real values appear later in the file).
NATIVE_NUMERIC_COLS = [
    'arp.opcode', 'data.len', 'dns.count.add_rr', 'dns.count.answers', 'dns.count.auth_rr',
    'dns.count.queries', 'dns.flags.authoritative', 'dns.flags.checkdisable', 'dns.flags.opcode',
    'dns.flags.response', 'eapol.keydes.key_len', 'eapol.keydes.replay_counter', 'eapol.len',
    'frame.encap_type', 'frame.len', 'http.next_request_in', 'http.next_response_in',
    'http.request_in', 'http.response.code', 'http.time', 'icmpv6.mldr.nb_mcast_records', 'ldap',
    'radiotap.channel.flags.cck', 'radiotap.channel.flags.ofdm', 'radiotap.channel.freq',
    'radiotap.datarate', 'radiotap.length', 'radiotap.mactime', 'radiotap.present.tsft',
    'tcp.ack', 'tcp.analysis.flags', 'tcp.analysis.rto_frame', 'tcp.dstport', 'tcp.flags.ack',
    'tcp.flags.fin', 'tcp.flags.push', 'tcp.flags.reset', 'tcp.flags.syn', 'tcp.seq', 'tcp.srcport',
    'tcp.time_delta', 'tcp.time_relative', 'udp.dstport', 'udp.length', 'udp.srcport',
    'wlan.duration', 'wlan.fc.frag', 'wlan.fc.moredata', 'wlan.fc.order', 'wlan.fc.protected',
    'wlan.fc.pwrmgt', 'wlan.fc.retry', 'wlan.fc.subtype', 'wlan.fc.type',
    'wlan.fixed.capabilities.ess', 'wlan.fixed.capabilities.ibss', 'wlan.fixed.timestamp',
    'wlan.rsn.capabilities.mfpc', 'wlan.seq', 'wlan_radio.channel', 'wlan_radio.data_rate',
    'wlan_radio.duration', 'wlan_radio.end_tsf', 'wlan_radio.frequency', 'wlan_radio.phy',
    'wlan_radio.signal_dbm', 'wlan_radio.start_tsf', 'wlan_radio.timestamp',
    'wlan_rsna_eapol.keydes.data_len', 'wlan_rsna_eapol.keydes.key_info.key_mic',
    'wlan_rsna_eapol.keydes.msgnr', 'wlan_rsna_eapol.keydes.nonce', 'wlan_rsna_eapol.keydes.data',
]
FEATURE_NAMES = sorted(set(NATIVE_NUMERIC_COLS) - set(DROP_ALWAYS) - set(LEAKAGE_COLS)) + COERCE_NUMERIC + HEX_NUMERIC
print(f'Feature candidates before zero-variance check: {len(FEATURE_NAMES)}')

## 1. Load Data (chunked, memory-efficient)
Pass 1 reads only the label column (fast) to get the class distribution and fit the
label encoder. Pass 2 reads the file again in 50,000-row chunks, cleaning and casting
each chunk to a small float32 array immediately -- the full raw DataFrame is never
held in memory at once.

In [ ]:
t0 = time.time()

# Pass 1: labels only (fast, low memory) -- also fits the label encoder up front
labels_only = pd.read_csv(RAW_CSV, usecols=[LABEL_COL])
print(f'Total rows : {len(labels_only):,}')
print('\nClass distribution (raw):')
dist = labels_only[LABEL_COL].value_counts()
print(dist.to_string())
print(f'\nImbalance ratio (max/min): {dist.max()/dist.min():.0f}:1')

le = LabelEncoder()
le.fit(labels_only[LABEL_COL])
del labels_only

In [ ]:
# Pass 2: chunked clean + feature extraction.
# Reading the whole 4.85M-row / 135-column CSV in one shot previously drove this
# process to ~13.7GB RSS and got OOM-killed (15GB RAM machine) -- this is what the
# original KeyboardInterrupt actually was. Reading in 50,000-row chunks and casting
# each chunk straight to float32 keeps peak memory in the hundreds of MB.
X_chunks, y_chunks, time_chunks = [], [], []

for i, chunk in enumerate(pd.read_csv(RAW_CSV, chunksize=CHUNK_SIZE)):
    for col in HEX_NUMERIC:
        chunk[col] = parse_hex(chunk[col])

    # Coerce + select FEATURE_NAMES in one shot; non-numeric leftovers (shouldn't
    # occur given NATIVE_NUMERIC_COLS/COERCE_NUMERIC/HEX_NUMERIC were validated
    # whole-file) and true missing values both become 0 ('field does not apply to
    # this packet type').
    feat = chunk[FEATURE_NAMES].apply(pd.to_numeric, errors='coerce').fillna(0)

    X_chunks.append(feat.values.astype(np.float32))
    y_chunks.append(le.transform(chunk[LABEL_COL]))
    time_chunks.append(chunk[TIME_KEY_COL].values.astype(np.float64))

    del chunk, feat
    if i % 4 == 0:
        print(f'  chunk {i+1} done | elapsed {time.time()-t0:.1f}s')

X_all = np.concatenate(X_chunks); del X_chunks
y_all = np.concatenate(y_chunks); del y_chunks
time_key_all = np.concatenate(time_chunks); del time_chunks
print(f'\nLoaded in {time.time()-t0:.1f}s  |  X_all shape: {X_all.shape}')

## 2. Zero-Variance Check
The only automatic column-dropping left: a feature that is exactly constant across
all 4.85M rows after the 0-fill (e.g. a field that quite literally never appears in
this dataset) carries zero information regardless of FS/FE method, so it's dropped
here. This is different from the missing-value-threshold drop removed above --
sparse-but-variable columns are kept; only truly constant ones go.

In [ ]:
variances = X_all.var(axis=0)
zero_var_mask = variances == 0
if zero_var_mask.any():
    zero_var_cols = [FEATURE_NAMES[j] for j in range(len(FEATURE_NAMES)) if zero_var_mask[j]]
    print(f'Dropping zero-variance columns ({len(zero_var_cols)}): {zero_var_cols}')
    keep_mask = ~zero_var_mask
    X_all = X_all[:, keep_mask]
    FEATURE_NAMES = [f for f, k in zip(FEATURE_NAMES, keep_mask) if k]
else:
    print('No zero-variance columns found.')

print(f'\nFinal feature count: {len(FEATURE_NAMES)}')
print('Features kept:')
for f in FEATURE_NAMES:
    print(f'  {f}')

## 3. Time-Based 70 / 15 / 15 Split (per class) — leakage fix

Random stratified row splitting let near-identical adjacent packets from the
same capture session land in both train and test, hugely inflating scores
(see diagnostic in the leakage-fix note above). Instead, **within each class**,
rows are sorted chronologically by `frame.time_epoch` and split along that
timeline: earliest 70% → train, next 15% → val, latest 15% → test. Indices are
then shuffled within each split (so classifiers don't see one time-ordered
block at a time), but the train/val/test *boundary* itself always falls at a
fixed point in time per class.

In [ ]:
train_idx_parts, val_idx_parts, test_idx_parts = [], [], []

for cls_i in range(len(le.classes_)):
    cls_idx = np.where(y_all == cls_i)[0]
    # Stable sort -> ties (same epoch second) keep original file/capture order
    order = cls_idx[np.argsort(time_key_all[cls_idx], kind='stable')]

    n = len(order)
    n_train = int(round(n * TRAIN_RATIO))
    n_val   = int(round(n * VAL_RATIO))

    train_idx_parts.append(order[:n_train])
    val_idx_parts.append(order[n_train:n_train + n_val])
    test_idx_parts.append(order[n_train + n_val:])

train_idx = np.concatenate(train_idx_parts)
val_idx   = np.concatenate(val_idx_parts)
test_idx  = np.concatenate(test_idx_parts)

# Shuffle row order WITHIN each split only (does not move rows across the time boundary)
rng = np.random.default_rng(RANDOM_STATE)
rng.shuffle(train_idx)
rng.shuffle(val_idx)
rng.shuffle(test_idx)

X_train, y_train = X_all[train_idx], y_all[train_idx]
X_val,   y_val   = X_all[val_idx],   y_all[val_idx]
X_test,  y_test  = X_all[test_idx],  y_all[test_idx]

print(f'Train : {X_train.shape}  |  {len(y_train):,} samples')
print(f'Val   : {X_val.shape}    |  {len(y_val):,} samples')
print(f'Test  : {X_test.shape}   |  {len(y_test):,} samples')

# Verify class distribution is maintained
print('\nClass distribution check (%):')
print(f"{'Class':<25} {'Train':>8} {'Val':>8} {'Test':>8}")
for i, cls in enumerate(le.classes_):
    tr = (y_train == i).mean() * 100
    va = (y_val   == i).mean() * 100
    te = (y_test  == i).mean() * 100
    print(f'{cls:<25} {tr:8.2f} {va:8.2f} {te:8.2f}')

# Sanity check: confirm the time boundary per class (train max epoch <= test min epoch)
print('\nPer-class time-boundary sanity check (train should end before test begins):')
for i, cls in enumerate(le.classes_):
    tr_mask, te_mask = (y_train == i), (y_test == i)
    if tr_mask.sum() == 0 or te_mask.sum() == 0:
        continue
    train_max = time_key_all[train_idx][tr_mask].max()
    test_min  = time_key_all[test_idx][te_mask].min()
    ok = '✓' if train_max <= test_min else '✗ OVERLAP'
    print(f'  {cls:<25} train_max_epoch={train_max:.0f}  test_min_epoch={test_min:.0f}  {ok}')

In [ ]:
print('Classes and their encoded integers:')
for i, cls in enumerate(le.classes_):
    count = (y_all == i).sum()
    print(f'  {i:2d}  {cls:<25}  n={count:>10,}')

print(f'\nX shape: {X_all.shape}')
print(f'y shape: {y_all.shape}')

## 4. StandardScaler
**Critical**: fit ONLY on training data to prevent data leakage.

In [ ]:
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)    # fit + transform on train
X_val_sc   = scaler.transform(X_val)           # transform only
X_test_sc  = scaler.transform(X_test)          # transform only

print('Scaler fitted on training set only.')
print(f'Train mean range: [{X_train_sc.mean(axis=0).min():.4f}, {X_train_sc.mean(axis=0).max():.4f}]')
print(f'Train std  range: [{X_train_sc.std(axis=0).min():.4f}, {X_train_sc.std(axis=0).max():.4f}]')

## 5. Save Artifacts

In [ ]:
# Save numpy arrays (compressed)
np.savez_compressed(f'{OUT_DIR}splits.npz',
    X_train=X_train_sc, X_val=X_val_sc, X_test=X_test_sc,
    y_train=y_train,    y_val=y_val,    y_test=y_test,
    # Also save unscaled (needed for tree-based models that don't require scaling)
    X_train_raw=X_train, X_val_raw=X_val, X_test_raw=X_test
)

# Save scaler and label encoder
with open(f'{OUT_DIR}scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
with open(f'{OUT_DIR}label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)

# Save metadata as JSON (human-readable)
meta = {
    'n_samples_total' : int(X_all.shape[0]),
    'n_features'      : int(X_all.shape[1]),
    'n_classes'       : int(len(le.classes_)),
    'classes'         : le.classes_.tolist(),
    'feature_names'   : FEATURE_NAMES,
    'split'           : {'train': int(len(y_train)), 'val': int(len(y_val)), 'test': int(len(y_test))},
    'split_strategy'  : 'time-based per class (earliest 70% -> train, next 15% -> val, latest 15% -> test); NOT random stratified',
    'random_state'    : RANDOM_STATE,
    'chunk_size_used' : CHUNK_SIZE,
    'dropped_always'  : DROP_ALWAYS,
    'dropped_leakage_features': LEAKAGE_COLS,
    'leakage_note'    : 'frame.time_epoch alone gave macro F1=1.0000 (session-identity leak); excluded from features, used only as the split ordering key',
    'coerced_numeric' : COERCE_NUMERIC,
    'nan_fill'        : 0,
    'preprocessing_policy': 'keep all native-numeric raw columns (no missing-value threshold drop); '
                             'fillna(0); drop only identifiers/free-text and zero-variance columns',
    'scaler'          : 'StandardScaler (fit on train only)',
    'class_counts_train': {
        cls: int((y_train == i).sum()) for i, cls in enumerate(le.classes_)
    }
}
with open(f'{OUT_DIR}meta.json', 'w') as f:
    json.dump(meta, f, indent=2)

print('Saved:')
for fn in ['splits.npz', 'scaler.pkl', 'label_encoder.pkl', 'meta.json']:
    path = f'{OUT_DIR}{fn}'
    size = os.path.getsize(path) / 1e6
    print(f'  {path}  ({size:.1f} MB)')

## 6. Summary Report

In [ ]:
# Class distribution bar chart
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

dist_df = pd.Series(dist.values, index=dist.index)
dist_df.plot.barh(ax=axes[0], color='steelblue')
axes[0].set_title('Raw class distribution')
axes[0].set_xlabel('Sample count')

dist_df_log = np.log10(dist_df + 1)
dist_df_log.plot.barh(ax=axes[1], color='tomato')
axes[1].set_title('Raw class distribution (log10 scale)')
axes[1].set_xlabel('log10(count)')

plt.tight_layout()
plt.savefig(f'{OUT_DIR}class_distribution.png', dpi=100)
plt.close()

print('=== PREPROCESSING SUMMARY ===')
print(f'  Total samples      : {X_all.shape[0]:,}')
print(f'  Features kept      : {len(FEATURE_NAMES)}  (4 leakage features excluded: {LEAKAGE_COLS})')
print(f'  Classes            : {len(le.classes_)}')
print(f'  Split strategy     : time-based per class (NOT random stratified) -- see leakage fix note')
print(f'  Train / Val / Test : {len(y_train):,} / {len(y_val):,} / {len(y_test):,}')
print(f'  Imbalance ratio    : {dist.max()/dist.min():.0f}:1  (handled in experiment notebooks)')
print(f'  Saved to           : {OUT_DIR}')
print()
print(f'Feature list: {FEATURE_NAMES}')